# One-step direct teacher-target regression

This notebook trains an unconditional CIFAR-10 student to predict a deterministic
diffusion teacher's final output from the same Gaussian noise. This is **direct
teacher-output regression**, not consistency distillation. The student uses the
same image-valued wrapper as the original notebook, but training only supervises
the timestep-998 output.

Run all cells in order on a fresh Google Colab GPU runtime. Google Drive must
contain `ddpm_cifar_300.pt` at the path below; edit the path if necessary.
The checkpoint SHA-256 is checked before loading. CIFAR-10, `pytorch-fid`, and
the FID Inception weights may require downloads.

The notebook generates 2,048 teacher-target training pairs, trains for 12,000
updates in one run, saves the model and optimizer, and then compares the student
and teacher on the **same 5,000 fresh noise inputs**. Both FIDs use the first
5,000 CIFAR-10 training images as the real reference. The test noise seed was not
used by the earlier runs, although the CIFAR reference and model design were
used during development. A 5,000-image FID is not directly comparable to a
50,000-image FID, and the score is not guaranteed to match earlier runs.

Every run creates a new Google Drive output folder. PNGs used for FID are stored
temporarily under `/content`; the training pairs, student checkpoint, and result
JSON are saved in the new Drive folder.

In [1]:
%pip install pytorch-fid==0.3.0

In [2]:
import hashlib
import json
import math
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from google.colab import drive
from PIL import Image
from pytorch_fid.fid_score import calculate_fid_given_paths

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Select a GPU runtime in Colab before running this notebook.")
print("Device:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

Device: Tesla T4
PyTorch: 2.14.1+cu130
Torchvision: 0.29.1+cu130


## Verify the teacher and define the models

In [3]:
drive.mount("/content/gdrive")
TEACHER_PATH = Path("/content/gdrive/My Drive/Queens/CISC473/ddpm_cifar_300.pt")
EXPECTED_TEACHER_SHA256 = "93089b1c124f894c59dbef7d45a9b6c75f8727e2f009efdeb227d7957b9a2a71"
if not TEACHER_PATH.is_file():
    raise FileNotFoundError(f"Teacher checkpoint not found: {TEACHER_PATH}. Update TEACHER_PATH.")

digest = hashlib.sha256()
with open(TEACHER_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(chunk)
if digest.hexdigest() != EXPECTED_TEACHER_SHA256:
    raise ValueError("The teacher checkpoint differs from the original experiment.")
print("Verified teacher checkpoint:", TEACHER_PATH)

Mounted at /content/gdrive
Verified teacher checkpoint: /content/gdrive/My Drive/Queens/CISC473/ddpm_cifar_300.pt


In [4]:
T = 1000
s = 8e-3
steps = torch.linspace(0, T, T + 1, dtype=torch.float64, device=device)
alpha_bar = torch.cos(((steps / T + s) / (1 + s)) * math.pi / 2) ** 2
alpha_bar = alpha_bar / alpha_bar[0]
betas = (1 - alpha_bar[1:] / alpha_bar[:-1]).clamp(min=1e-5, max=0.999).float()
alphas = 1.0 - betas
alpha_time_t = torch.cumprod(alphas, dim=0)
if abs(alpha_time_t.double().sum().item() - 495.5774450877395) >= 0.001:
    raise ValueError("The teacher noise schedule differs from the original experiment.")

In [5]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb_scale = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb_scale)
        emb = t.float()[:, None] * emb[None, :]
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)


class block(nn.Module):
    def __init__(self, in_channels, out_channels, num_groups=8):
        super().__init__()
        self.normalize = nn.GroupNorm(num_groups, in_channels)
        self.convolute = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.act_func = nn.SiLU()

    def forward(self, x):
        return self.convolute(self.act_func(self.normalize(x)))


class resnetblock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()
        self.mlp = nn.Linear(time_dim, out_channels)
        self.block1 = block(in_channels, out_channels)
        self.block2 = block(out_channels, out_channels)
        self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, time_emb):
        h = self.block1(x)
        time_emb = self.mlp(time_emb)[:, :, None, None]
        h = self.block2(h + time_emb)
        return h + self.residual(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, time_dim=128):
        super().__init__()
        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.input_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)
        self.downblock1 = resnetblock(base_channels, base_channels, time_dim)
        self.downsample1 = nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.downblock2 = resnetblock(base_channels * 2, base_channels * 2, time_dim)
        self.downsample2 = nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.bottleneck1 = resnetblock(base_channels * 4, base_channels * 4, time_dim)
        self.bottleneck2 = resnetblock(base_channels * 4, base_channels * 4, time_dim)
        self.upsample1 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.upblock1 = resnetblock(base_channels * 4, base_channels * 2, time_dim)
        self.upsample2 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.upblock2 = resnetblock(base_channels * 2, base_channels, time_dim)
        self.output = nn.Conv2d(base_channels, in_channels, kernel_size=1)

    def forward(self, x, t):
        t_emb = self.time_embedding(t)
        x = self.input_conv(x)
        d1 = self.downblock1(x, t_emb)
        x = self.downsample1(d1)
        d2 = self.downblock2(x, t_emb)
        x = self.downsample2(d2)
        x = self.bottleneck1(x, t_emb)
        x = self.bottleneck2(x, t_emb)
        x = self.upsample1(x)
        x = torch.cat([x, d2], dim=1)
        x = self.upblock1(x, t_emb)
        x = self.upsample2(x)
        x = torch.cat([x, d1], dim=1)
        x = self.upblock2(x, t_emb)
        return self.output(x)

In [6]:
class ConsistencyStudent(nn.Module):
    def __init__(self, alpha_cumprod, sigma_data=0.5):
        super().__init__()
        self.unet = UNet()
        self.register_buffer('noise_std', torch.sqrt(1 - alpha_cumprod.clone()))
        self.sigma_data = sigma_data

    def forward(self, x, t):
        if x.ndim != 4 or x.shape[1:] != (3, 32, 32):
            raise ValueError('x must have shape [batch, 3, 32, 32].')
        if t.dtype != torch.long or t.ndim != 1 or t.shape[0] != x.shape[0]:
            raise ValueError('t must be a torch.long vector with one index per image.')
        if x.device != self.noise_std.device or t.device != x.device:
            raise ValueError('x, t, and the student must be on the same device.')
        if torch.any(t < 0) or torch.any(t >= self.noise_std.numel()):
            raise ValueError('timestep indices must be between 0 and 999.')
        sigma = self.noise_std[t].view(-1, 1, 1, 1)
        offset = sigma - self.noise_std[0]
        c_skip = self.sigma_data ** 2 / (offset ** 2 + self.sigma_data ** 2)
        c_out = self.sigma_data * offset / torch.sqrt(sigma ** 2 + self.sigma_data ** 2)
        return c_skip * x + c_out * self.unet(x, t)

In [7]:
teacher = UNet().to(device)
teacher_weights = torch.load(TEACHER_PATH, map_location=device, weights_only=True)
print(teacher.load_state_dict(teacher_weights))
del teacher_weights
teacher.eval()
teacher.requires_grad_(False)
print('Teacher parameters:', sum(p.numel() for p in teacher.parameters()))

torch.manual_seed(0)
student = ConsistencyStudent(alpha_time_t).to(device)
student.unet.load_state_dict(teacher.state_dict())
student.train()
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)
batch_rng = torch.Generator(device="cpu").manual_seed(56789)
print("Student U-Net initialized from the teacher.")

<All keys matched successfully>
Teacher parameters: 4789699
Student U-Net initialized from the teacher.


## Generate teacher-target training pairs

In [8]:
levels = torch.linspace(0, 998, steps=101).round().long().tolist()
assert levels[-3:] == [978, 988, 998]
assert all(a < b for a, b in zip(levels, levels[1:]))
substep_path = list(range(998, 987, -1)) + levels[-3::-1]
assert substep_path[:11] == list(range(998, 987, -1))
assert substep_path[-1] == 0
print("Teacher evaluations per image:", len(substep_path) - 1)


@torch.no_grad()
def direct_teacher_sample(noise, path):
    x = noise.to(device)

    for high, low in zip(path[:-1], path[1:]):
        t = torch.full((len(x),), high, dtype=torch.long, device=device)
        eps = teacher(x, t)
        a_high = alpha_time_t[high]
        a_low = alpha_time_t[low]

        x0_pred = (x - (1 - a_high).sqrt() * eps) / a_high.sqrt()
        x = a_low.sqrt() * x0_pred + (1 - a_low).sqrt() * eps

        if not torch.isfinite(x).all():
            raise FloatingPointError(f"Non-finite teacher output at {high} -> {low}")

    return x.cpu()


def make_noise(count, seed):
    generator = torch.Generator(device="cpu").manual_seed(seed)
    return torch.randn(count, 3, 32, 32, generator=generator)

Teacher evaluations per image: 109


In [9]:
TRAIN_PAIRS = 2048
TRAIN_SEED = 23456
BATCH_SIZE = 32
TRAIN_STEPS = 12000
TEST_IMAGES = 5000
TEST_SEED = 89012
if TEST_SEED == TRAIN_SEED:
    raise ValueError("Training and test noise seeds must differ.")

run_dir = Path(tempfile.mkdtemp(
    prefix="direct_teacher_target_final_", dir=TEACHER_PATH.parent
))
print("New output folder:", run_dir)

train_noise = make_noise(TRAIN_PAIRS, TRAIN_SEED)
target_batches = []
print(f"Generating {TRAIN_PAIRS} teacher-target training pairs...")
for start in range(0, TRAIN_PAIRS, BATCH_SIZE):
    batch = train_noise[start:start + BATCH_SIZE]
    target_batches.append(direct_teacher_sample(batch, substep_path))
    done = start + len(batch)
    if done % 256 == 0 or done == TRAIN_PAIRS:
        print(f"  {done}/{TRAIN_PAIRS}")

train_targets = torch.cat(target_batches)
if train_targets.shape != train_noise.shape or not torch.isfinite(train_targets).all():
    raise FloatingPointError("Teacher training targets have an invalid shape or contain non-finite values.")

train_pairs_path = run_dir / "train_pairs.pt"
with open(train_pairs_path, "xb") as f:
    torch.save({
        "noise": train_noise,
        "targets": train_targets,
        "seed": TRAIN_SEED,
        "teacher_sha256": EXPECTED_TEACHER_SHA256,
        "teacher_path": substep_path,
    }, f)
print("Saved training pairs:", train_pairs_path)

New output folder: /content/gdrive/My Drive/Queens/CISC473/direct_teacher_target_final_a_szg2lh
Generating 2048 teacher-target training pairs...
  256/2048
  512/2048
  768/2048
  1024/2048
  1280/2048
  1536/2048
  1792/2048
  2048/2048
Saved training pairs: /content/gdrive/My Drive/Queens/CISC473/direct_teacher_target_final_a_szg2lh/train_pairs.pt


## Train the student in one run

In [10]:
print(f"Training for {TRAIN_STEPS} updates...")
training_start = time.time()
window_loss = 0.0
for step in range(1, TRAIN_STEPS + 1):
    indices = torch.randint(len(train_noise), (BATCH_SIZE,), generator=batch_rng)
    x = train_noise[indices].to(device)
    target = train_targets[indices].to(device)
    t = torch.full((len(x),), 998, dtype=torch.long, device=device)

    prediction = student(x, t)
    loss = F.mse_loss(prediction, target)
    if not torch.isfinite(loss):
        raise FloatingPointError(f"Non-finite training loss at step {step}")

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0, error_if_nonfinite=True)
    optimizer.step()

    window_loss += loss.item()
    if step % 500 == 0 or step == TRAIN_STEPS:
        window_size = 500 if step % 500 == 0 else step % 500
        print(f"  step {step}/{TRAIN_STEPS}: mean batch MSE={window_loss / window_size:.6f}")
        window_loss = 0.0

training_seconds = time.time() - training_start
model_config = {
    "method": "direct teacher-output regression",
    "teacher_checkpoint": str(TEACHER_PATH),
    "teacher_sha256": EXPECTED_TEACHER_SHA256,
    "alpha_time_t": alpha_time_t.cpu(),
    "teacher_path": substep_path,
    "sampling_timestep": 998,
    "training_pairs": TRAIN_PAIRS,
    "training_noise_seed": TRAIN_SEED,
    "batch_seed": 56789,
    "batch_size": BATCH_SIZE,
    "learning_rate": 1e-4,
    "gradient_clip_norm": 1.0,
    "training_steps": TRAIN_STEPS,
    "test_noise_seed": TEST_SEED,
    "test_images": TEST_IMAGES,
    "sigma_data": student.sigma_data,
}
checkpoint_path = run_dir / "final_student.pt"
with open(checkpoint_path, "xb") as f:
    torch.save({
        "step": TRAIN_STEPS,
        "student": student.state_dict(),
        "optimizer": optimizer.state_dict(),
        "batch_generator_state": batch_rng.get_state(),
        "config": model_config,
        "training_seconds": training_seconds,
    }, f)
print(f"Training finished in {training_seconds / 60:.1f} minutes.")
print("Saved student and optimizer:", checkpoint_path)

Training for 12000 updates...
  step 500/12000: mean batch MSE=0.084685
  step 1000/12000: mean batch MSE=0.008596
  step 1500/12000: mean batch MSE=0.005874
  step 2000/12000: mean batch MSE=0.004439
  step 2500/12000: mean batch MSE=0.003496
  step 3000/12000: mean batch MSE=0.002891
  step 3500/12000: mean batch MSE=0.002420
  step 4000/12000: mean batch MSE=0.002094
  step 4500/12000: mean batch MSE=0.001820
  step 5000/12000: mean batch MSE=0.001599
  step 5500/12000: mean batch MSE=0.001483
  step 6000/12000: mean batch MSE=0.001287
  step 6500/12000: mean batch MSE=0.001140
  step 7000/12000: mean batch MSE=0.001071
  step 7500/12000: mean batch MSE=0.000988
  step 8000/12000: mean batch MSE=0.000900
  step 8500/12000: mean batch MSE=0.000793
  step 9000/12000: mean batch MSE=0.000778
  step 9500/12000: mean batch MSE=0.000759
  step 10000/12000: mean batch MSE=0.000648
  step 10500/12000: mean batch MSE=0.000676
  step 11000/12000: mean batch MSE=0.000570
  step 11500/12000: me

## Evaluate student and teacher on matched fresh noise

In [11]:
fid_root = Path(tempfile.mkdtemp(prefix="direct_teacher_fid_", dir="/content"))
real_dir = fid_root / "real"
student_dir = fid_root / "student"
teacher_dir = fid_root / "teacher"
for folder in (real_dir, student_dir, teacher_dir):
    folder.mkdir()

real_dataset = torchvision.datasets.CIFAR10(
    root="/content/data", train=True, download=True
)
if len(real_dataset) < TEST_IMAGES:
    raise ValueError("The CIFAR-10 reference split has fewer images than requested.")

print(f"Saving {TEST_IMAGES} real CIFAR-10 training images...")
for i in range(TEST_IMAGES):
    image, _ = real_dataset[i]
    image.save(real_dir / f"{i:05d}.png")

test_rng = torch.Generator(device="cpu").manual_seed(TEST_SEED)
student.eval()
student_below = 0
student_above = 0
teacher_below = 0
teacher_above = 0
pixel_count = 0
squared_error_sum = 0.0
print(f"Generating {TEST_IMAGES} matched student and teacher images...")
with torch.inference_mode():
    for start in range(0, TEST_IMAGES, BATCH_SIZE):
        count = min(BATCH_SIZE, TEST_IMAGES - start)
        noise = torch.randn(count, 3, 32, 32, generator=test_rng)
        t = torch.full((count,), 998, dtype=torch.long, device=device)
        student_output = student(noise.to(device), t).cpu()
        teacher_output = direct_teacher_sample(noise, substep_path)
        if student_output.shape != noise.shape or teacher_output.shape != noise.shape:
            raise ValueError(f"Invalid output shape at test image {start}")
        if not torch.isfinite(student_output).all() or not torch.isfinite(teacher_output).all():
            raise FloatingPointError(f"Non-finite output at test image {start}")

        student_below += (student_output < 0).sum().item()
        student_above += (student_output > 1).sum().item()
        teacher_below += (teacher_output < 0).sum().item()
        teacher_above += (teacher_output > 1).sum().item()
        pixel_count += student_output.numel()
        squared_error_sum += (student_output - teacher_output).square().sum().item()

        student_pixels = student_output.clamp(0, 1).mul(255).round().to(torch.uint8)
        teacher_pixels = teacher_output.clamp(0, 1).mul(255).round().to(torch.uint8)
        for j in range(count):
            name = f"{start + j:05d}.png"
            Image.fromarray(student_pixels[j].permute(1, 2, 0).numpy()).save(student_dir / name)
            Image.fromarray(teacher_pixels[j].permute(1, 2, 0).numpy()).save(teacher_dir / name)

        done = start + count
        if done % 1024 == 0 or done == TEST_IMAGES:
            print(f"  {done}/{TEST_IMAGES}")

for folder in (real_dir, student_dir, teacher_dir):
    if len(list(folder.glob("*.png"))) != TEST_IMAGES:
        raise RuntimeError(f"Incorrect number of images in {folder}")
print("FID images prepared:", fid_root)

100%|██████████| 170M/170M [13:42<00:00, 207kB/s]  


Saving 5000 real CIFAR-10 training images...
Generating 5000 matched student and teacher images...
  1024/5000
  2048/5000
  3072/5000
  4096/5000
  5000/5000
FID images prepared: /content/direct_teacher_fid_ieylpy9l


In [12]:
print("Calculating student FID...")
student_fid = calculate_fid_given_paths(
    [str(real_dir), str(student_dir)], batch_size=50, device=device, dims=2048
)
print("Calculating teacher FID...")
teacher_fid = calculate_fid_given_paths(
    [str(real_dir), str(teacher_dir)], batch_size=50, device=device, dims=2048
)
if not math.isfinite(student_fid) or not math.isfinite(teacher_fid):
    raise FloatingPointError("A calculated FID is not finite.")

digest = hashlib.sha256()
with open(checkpoint_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(chunk)

results = {
    "method": "direct teacher-output regression",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "student_fid": float(student_fid),
    "teacher_fid": float(teacher_fid),
    "paired_raw_mse": squared_error_sum / pixel_count,
    "test_images": TEST_IMAGES,
    "test_noise_seed": TEST_SEED,
    "real_reference": f"first {TEST_IMAGES} CIFAR-10 training images",
    "real_preprocessing": "original CIFAR-10 uint8 PNG",
    "generated_preprocessing": "clamp [0,1], round to uint8 PNG",
    "fid_implementation": "pytorch-fid 0.3.0, Inception features 2048",
    "student_evaluations_per_image": 1,
    "teacher_evaluations_per_image": len(substep_path) - 1,
    "student_fraction_below_zero": student_below / pixel_count,
    "student_fraction_above_one": student_above / pixel_count,
    "teacher_fraction_below_zero": teacher_below / pixel_count,
    "teacher_fraction_above_one": teacher_above / pixel_count,
    "teacher_sha256": EXPECTED_TEACHER_SHA256,
    "student_checkpoint": str(checkpoint_path),
    "student_checkpoint_sha256": digest.hexdigest(),
    "training_pairs": TRAIN_PAIRS,
    "training_noise_seed": TRAIN_SEED,
    "training_steps": TRAIN_STEPS,
    "training_batch_size": BATCH_SIZE,
    "learning_rate": 1e-4,
    "training_seconds": training_seconds,
    "torch_version": str(torch.__version__),
    "torchvision_version": str(torchvision.__version__),
    "gpu": torch.cuda.get_device_name(0),
}
result_path = run_dir / "fid_results.json"
with open(result_path, "x") as f:
    json.dump(results, f, indent=2, allow_nan=False)

print(f"\nStudent FID ({TEST_IMAGES} images): {student_fid:.3f}")
print(f"Teacher FID ({TEST_IMAGES} images): {teacher_fid:.3f}")
print(f"Matched raw pixel MSE: {results['paired_raw_mse']:.6f}")
print(f"Student below/above [0,1]: {student_below / pixel_count:.2%} / {student_above / pixel_count:.2%}")
print(f"Teacher below/above [0,1]: {teacher_below / pixel_count:.2%} / {teacher_above / pixel_count:.2%}")
print("Saved checkpoint:", checkpoint_path)
print("Saved FID results:", result_path)

Calculating student FID...
Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:00<00:00, 180MB/s]
100%|██████████| 100/100 [00:21<00:00,  4.66it/s]


Calculating teacher FID...


100%|██████████| 100/100 [00:21<00:00,  4.59it/s]



Student FID (5000 images): 111.768
Teacher FID (5000 images): 53.174
Matched raw pixel MSE: 0.003358
Student below/above [0,1]: 9.90% / 0.94%
Teacher below/above [0,1]: 10.71% / 1.07%
Saved checkpoint: /content/gdrive/My Drive/Queens/CISC473/direct_teacher_target_final_a_szg2lh/final_student.pt
Saved FID results: /content/gdrive/My Drive/Queens/CISC473/direct_teacher_target_final_a_szg2lh/fid_results.json
